# NPTEL Study Notes: Knowledge Graph Embeddings (Translation & Rotation Models)

## 1. Introduction: The Scoring Function $f(s, r, o)$
In knowledge graphs (KGs), facts are represented as triples: **(Subject, Relation, Object)** or $(s, r, o)$. The goal of knowledge graph embedding models is to design a **scoring function $f(s, r, o)$** that outputs a "belief" or confidence score for a given triple.
*   While deep neural networks are universal function approximators and could technically learn this mapping, they are computationally heavy.
*   Instead, researchers design specific geometric operations (translations, rotations) in a vector space to represent relations, learning the optimal **embeddings** (dense vector representations) for entities and relations.

---

## 2. Background: Word Embeddings and the Translation Property
Models like **Word2Vec** and **GloVe** (based on low-rank matrix factorizations) established the foundational idea that semantic relationships can be captured as vector translations.
*   **Classic Example:** $\vec{Man} \rightarrow \vec{King}$ is approximately the same translation vector as $\vec{Woman} \rightarrow \vec{Queen}$. The translation vector represents the concept of "royalty".
*   **Knowledge Graph Parallel:** In a KG, entities of the same type related by the same relation often exhibit parallel translations. For example, the translation vector from a country to its capital (e.g., $Spain \rightarrow Madrid$) is nearly identical to ($Italy \rightarrow Rome$). This vector represents the relation `has_capital`.

---

## 3. TransE (Translation Embeddings)
TransE is one of the earliest and simplest models for KG embeddings. It models relations as simple vector translations in a $d$-dimensional real space ($\mathbb{R}^d$).

### 3.1. Mechanism and Scoring Function
*   **Core Idea:** If $(s, r, o)$ is a true fact, then the subject vector translated by the relation vector should be very close to the object vector: 
    $$s + r \approx o$$
*   **Scoring Function:** The score is defined as the distance (norm) between $(s + r)$ and $o$.
    $$f(s, r, o) = \vert{}\vert{}s + r - o\vert{}\vert{}$$
*   **Interpretation:** 
    *   A **small** distance (low score) means high confidence/belief that the triple is a valid fact.
    *   A **large** distance (high score) means low confidence.

### 3.2. Training and Loss Function
*   **Variables:** The function $f$ has *no internal network parameters*. The only variables optimized during training are the entity embeddings (subjects/objects) and relation embeddings.
*   **Loss Function:** A **hinge loss** with negative sampling is used.
    *   We want distances for *true* (positive) triples to be small.
    *   We want distances for *corrupted* (negative) triples to be large.
    *   The model takes $k$ negative samples per positive sample and adjusts the parameters to push the positive objects closer and the negative objects away.

### 3.3. Pros and Cons (Important for Exams)
*   **Advantages:**
    *   Extremely simple and fast computation (just addition, subtraction, and a norm).
    *   Very few hyperparameters (only the margin for hinge loss and $k$ negative samples).
*   **Limitations (Critical NPTEL Focus):**
    *   **Cannot model 1-to-N, N-to-1, or N-to-N relationships:** If Obama ($s$) `attended` ($r$) both Occidental College ($o_1$) and Harvard ($o_2$), TransE forces $s + r \approx o_1$ and $s + r \approx o_2$. The optimization will erroneously push $o_1$ and $o_2$ to the exact same point in space, implying they are the same entity.
    *   **Cannot model Symmetric relationships:** If a relation is symmetric (e.g., `is_sibling_of`), then $s + r = o$ and $o + r = s$. Solving these simultaneously yields $2r = 0 \implies r = 0$. Thus, TransE forces all symmetric relations to be the zero vector, collapsing them.

---

## 4. TransH (Translation on Hyperplanes)
To fix the structural limitations of TransE, variants like **TransH** were developed. TransH allows entities to have different representations depending on the specific relation they are involved in.

### 4.1. Mechanism
Instead of a single vector, TransH represents a relation $r$ using **two artifacts**:
1.  **$p_r$:** A unit normal vector that defines a relation-specific hyperplane.
2.  **$d_r$:** A displacement (translation) vector *within* that hyperplane.

### 4.2. Mathematical Operation
1.  First, project the subject $s$ and object $o$ onto the relation's hyperplane (defined by $p_r$).
2.  The translation $d_r$ is then applied between these projected points:
    $$s_{\text{projected}} + d_r \approx o_{\text{projected}}$$

### 4.3. Why it fixes TransE's Limitations
*   **Solves Many-to-Many / 1-to-N:** Barack Obama ($s$) can project down to a specific point on the `attended` hyperplane. Both Occidental College ($o_1$) and Harvard ($o_2$) can exist at different points in the overall vector space, but they project down to the *same* appropriate region on the `attended` hyperplane. This allows 1-to-N mappings without forcing $o_1$ and $o_2$ to be identical in the absolute space.

---

## 5. RotatE (Rotation in Complex Space)
RotatE replaces simple real-number translations with **rotations in a complex vector space** ($\mathbb{C}^d$).

### 5.1. Complex Vector Refresher
*   A complex number $z$ is represented as $a + jb$, where $j = \sqrt{-1}$.
*   In polar coordinates, $z = R e^{j\theta} = R(\cos\theta + j\sin\theta)$.
*   Multiplying a complex number by another complex number of unit length ($r = 1 e^{j\theta}$) results in **rotating** the original vector by the angle $\theta$ counterclockwise.
*   **Complex Norm:** For an entity vector with $d$ complex numbers, the norm is:
    $$\vert{}\vert{}z\vert{}\vert{} = \sqrt{\sum_{i=1}^{d} (a_i^2 + b_i^2)}$$
    *(Note: A $d$-dimensional complex vector is represented by $2d$ real numbers).*

### 5.2. Mechanism and Scoring Function
*   Entities ($s, o$) are embedded as $d$-dimensional complex vectors.
*   Relations ($r$) are embedded as complex vectors where *each dimension has a unit modulus (norm = 1)*.
*   **Operation:** The subject is rotated by the relation via element-wise complex multiplication (denoted here as $\circ$):
    $$s \circ r \approx o$$
*   **Scoring Function:** 
    $$f(s, r, o) = \vert{}\vert{}s \circ r - o\vert{}\vert{}^2$$

### 5.3. Capabilities and Properties (Exam Focus)
RotatE is a superset of TransE (at least as powerful) and gracefully handles complex relationship topologies:
1.  **Symmetric Relations:** If $(s, r, o)$ and $(o, r, s)$ are both true, then $o = r \circ s$ and $s = r \circ o$. Substituting gives $r \circ r = 1$. The only way a rotation acts as its own inverse is if $\theta = 180^\circ$ ($\pi$ radians). Thus, symmetric relations are modeled naturally as $180^\circ$ rotations. *Crucially, unlike TransE which collapsed all symmetric relations to 0, RotatE allows different symmetric relations to have different $180^\circ$ rotations across the $d$ dimensions.*
2.  **Anti-symmetric Relations:** Modeled by rotation angles other than $180^\circ$.
3.  **Inverse Relations:** If $r$ and $r'$ are inverses, one is a clockwise rotation ($-\theta$) and the other is an anticlockwise rotation ($+\theta$).
4.  **Compositional Relations:** E.g., `father_of` + `born_in` = `father_born_in`. Because RotatE uses rotations, composing relations is mathematically elegant: you simply **add the angles of rotation** ($\theta_1 + \theta_2$) to traverse the graph.

---

## 6. Practice Questions

**Q1. In the TransE model, why is it problematic to model the symmetric relationship `is_friend_of`?**
a) The model requires an infinite number of negative samples.
b) The relation vector $r$ is forced to become the zero vector, making all symmetric relations indistinguishable.
c) The relation vector $r$ rotates the subject by 180 degrees, which is computationally expensive.
d) The hyperplanes for symmetric relations become parallel.
*   **Answer:** (b)
*   **Explanation:** For symmetry, $s+r=o$ and $o+r=s$. Solving gives $2r=0$, so $r=0$.

**Q2. How does TransH overcome the inability of TransE to model 1-to-N relationships?**
a) By using complex numbers to rotate the subject entity.
b) By projecting the subject and object entities onto a relation-specific hyperplane before translating.
c) By assigning multiple relation vectors for every single fact in the knowledge graph.
d) By increasing the margin in the hinge loss function.
*   **Answer:** (b)
*   **Explanation:** TransH uses a unit normal $p_r$ to define a hyperplane and $d_r$ for translation. Entities project onto this plane, allowing distinct entities in 3D space to map to similar points in the relation's sub-space.

**Q3. In the RotatE model, if an entity is represented in a 5-dimensional complex space ($d=5$), how many actual real numbers are stored for this entity embedding?**
a) 5
b) 10
c) 15
d) 25
*   **Answer:** (b)
*   **Explanation:** Every complex number has a real and an imaginary part ($a+jb$). For $d=5$, there are $5 \times 2 = 10$ real numbers.

**Q4. In the RotatE model, what is a strict requirement for the relation embeddings during gradient descent optimization?**
a) The sum of the dimensions must equal zero.
b) Each complex element of the relation vector must have a unit modulus (magnitude of 1).
c) The relation vector must only contain real numbers.
d) The relation vector must be orthogonal to the subject vector.
*   **Answer:** (b)
*   **Explanation:** Because the relation is meant to act strictly as a *rotation* (and not scaling), multiplying by it must not change the length of the vector. Thus, $r = 1 e^{j\theta}$, requiring unit modulus.

---

## 7. Revision Summary
*   **Objective:** Score KG triples $(s,r,o)$ so true facts have low distance (high belief) and false facts have high distance. Models use Hinge loss + negative sampling.
*   **TransE:** $s+r \approx o$. Simple and fast, but fails on N-to-N, 1-to-N, and symmetric relationships (forces $r=0$).
*   **TransH:** Projects entities onto a relation-specific hyperplane ($p_r$) before translating by $d_r$. Fixes 1-to-N mapping issues.
*   **RotatE:** Uses complex space ($\mathbb{C}^d$). $s \circ r \approx o$. Relation must have unit norm ($\vert{}\vert{}r\vert{}\vert{}=1$). Elegantly handles symmetry ($180^\circ$ rotation), inverse relations ($-\theta$), and relation composition (adding angles).
```eof